# Task 3: Citation Span Extraction - BERT QA

**Model:** bert-base-uncased (Question Answering)

**Task:** Extract text span that citation supports

**KEY FEATURES:**
- ✅ Character-level metrics (accurate evaluation)
- ✅ Weights & Biases tracking (experiment management)
- ✅ Pre-computed s_span/e_span positions
- ✅ F1 + Exact Match on extracted text
- ✅ Early stopping

---

## 1. Setup & Imports

In [1]:
import transformers, datasets, accelerate
print(f"✅ transformers: {transformers.__version__}")
print(f"✅ datasets: {datasets.__version__}")
print(f"✅ accelerate: {accelerate.__version__}")

✅ transformers: 5.0.0
✅ datasets: 4.8.3
✅ accelerate: 1.12.0


## 2. Weights & Biases Setup

In [2]:
import wandb
from kaggle_secrets import UserSecretsClient

try:
    secrets = UserSecretsClient()
    key = secrets.get_secret("WANDB_API_KEY")

    result = wandb.login(key=key)

    if result:
        print("✅ Wandb logged in")
    else:
        print("⚠️ Wandb login returned False - API key có thể sai")

except Exception as e:
    print(f"⚠️ Wandb login failed: {type(e).__name__}: {e}")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: tathiyennhi (tathiyennhi-hcmus) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✅ Wandb logged in


## 3. Check Dataset

In [3]:
import os

train_path = "/kaggle/input/datasets/tathiyennhi/task3-citation-span-extraction/task3/train"
val_path = "/kaggle/input/datasets/tathiyennhi/task3-citation-span-extraction/task3/val"

train_count = len([f for f in os.listdir(train_path) if f.endswith('.label')])
val_count = len([f for f in os.listdir(val_path) if f.endswith('.label')])

print(f"✅ Train: {train_count:,} files")
print(f"✅ Val: {val_count:,} files")

✅ Train: 54,356 files
✅ Val: 2,910 files


## 4. Load Data

In [4]:
import json
from pathlib import Path
from datasets import Dataset

def load_task3_data(data_dir, max_examples=None):
    """
    Load data and return as list (for non-streaming dataset)
    """
    data_path = Path(data_dir)
    label_files = sorted(data_path.glob("*.label"))
    
    if max_examples:
        label_files = label_files[:max_examples]
    
    total_files = len(label_files)
    print(f"📊 Loading {total_files:,} files from {data_dir}")
    
    examples = []
    skipped = 0

    for i, label_file in enumerate(label_files):
        if (i+1) % 5000 == 0:
            print(f"⏳ {i+1:,}/{total_files:,} | Loaded: {len(examples):,} | Skipped: {skipped}")

        try:
            with open(label_file) as f:
                label_data = json.load(f)
        except:
            skipped += 1
            continue

        text = label_data.get('text', '')
        if not text:
            skipped += 1
            continue
            
        citation_spans = label_data.get('citation_spans', [])

        for span_info in citation_spans:
            citation_id = span_info.get('citation_id', '')
            span_text = span_info.get('span_text', '')
            s_span = span_info.get('s_span', -1)
            e_span = span_info.get('e_span', -1)
            
            if s_span == -1 or e_span == -1 or s_span >= e_span:
                skipped += 1
                continue

            question = f"What does citation {citation_id} support?"
            
            examples.append({
                'question': question,
                'context': text,
                'answer_text': span_text,
                'answer_start_char': s_span,
                'answer_end_char': e_span
            })

    print(f"✅ Loaded {len(examples):,} examples | Skipped: {skipped}")
    return examples

# Load data
print("=" * 60)
train_examples = load_task3_data(train_path)
val_examples = load_task3_data(val_path)

# Convert to Dataset
train_dataset = Dataset.from_list(train_examples)
val_dataset = Dataset.from_list(val_examples)

print(f"\n✅ Train dataset: {len(train_dataset):,} examples")
print(f"✅ Val dataset: {len(val_dataset):,} examples")

📊 Loading 54,356 files from /kaggle/input/datasets/tathiyennhi/task3-citation-span-extraction/task3/train
⏳ 5,000/54,356 | Loaded: 11,791 | Skipped: 0
⏳ 10,000/54,356 | Loaded: 23,469 | Skipped: 0
⏳ 15,000/54,356 | Loaded: 34,981 | Skipped: 0
⏳ 20,000/54,356 | Loaded: 46,313 | Skipped: 0
⏳ 25,000/54,356 | Loaded: 57,749 | Skipped: 0
⏳ 30,000/54,356 | Loaded: 69,736 | Skipped: 0
⏳ 35,000/54,356 | Loaded: 81,576 | Skipped: 0
⏳ 40,000/54,356 | Loaded: 93,307 | Skipped: 0
⏳ 45,000/54,356 | Loaded: 104,943 | Skipped: 0
⏳ 50,000/54,356 | Loaded: 116,406 | Skipped: 0
✅ Loaded 126,622 examples | Skipped: 0
📊 Loading 2,910 files from /kaggle/input/datasets/tathiyennhi/task3-citation-span-extraction/task3/val
✅ Loaded 6,894 examples | Skipped: 0

✅ Train dataset: 126,622 examples
✅ Val dataset: 6,894 examples


## 5. Tokenization

In [5]:
from transformers import AutoTokenizer

MODEL_PATH = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
print(f"✅ Tokenizer loaded from: {MODEL_PATH}")


def prepare_features(examples):
    tokenized = tokenizer(
        examples["question"],
        examples["context"],
        max_length=512,
        truncation="only_second",
        padding="max_length",
        return_offsets_mapping=True,
    )

    start_positions = []
    end_positions = []

    for i in range(len(examples["question"])):
        start_char = examples["answer_start_char"][i]
        end_char = examples["answer_end_char"][i]

        offsets = tokenized["offset_mapping"][i]
        sequence_ids = tokenized.sequence_ids(i)

        # chỉ lấy phần context (sequence_id = 1)
        context_start = 0
        while sequence_ids[context_start] != 1:
            context_start += 1

        context_end = len(sequence_ids) - 1
        while sequence_ids[context_end] != 1:
            context_end -= 1

        # nếu answer nằm ngoài context → set (0,0)
        if not (offsets[context_start][0] <= start_char and offsets[context_end][1] >= end_char):
            start_positions.append(0)
            end_positions.append(0)
            continue

        # tìm start token
        start_token = context_start
        for idx in range(context_start, context_end + 1):
            if offsets[idx][0] <= start_char < offsets[idx][1]:
                start_token = idx
                break

        # tìm end token
        end_token = context_end
        for idx in range(context_start, context_end + 1):
            if offsets[idx][0] < end_char <= offsets[idx][1]:
                end_token = idx
                break

        start_positions.append(start_token)
        end_positions.append(end_token)

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions

    return tokenized


print("Tokenizing train dataset...")
train_dataset = train_dataset.map(
    prepare_features,
    batched=True,
    remove_columns=[
        "question",
        "context",
        "answer_text",
        "answer_start_char",
        "answer_end_char",
    ],
)

print("Tokenizing val dataset...")
val_dataset = val_dataset.map(
    prepare_features,
    batched=True,
    remove_columns=[
        "question",
        "context",
        "answer_text",
        "answer_start_char",
        "answer_end_char",
    ],
)

print("\n✅ Tokenization complete")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Tokenizer loaded from: bert-base-uncased
Tokenizing train dataset...


Map:   0%|          | 0/126622 [00:00<?, ? examples/s]

Tokenizing val dataset...


Map:   0%|          | 0/6894 [00:00<?, ? examples/s]


✅ Tokenization complete


## 6. Character-Level Metrics

In [6]:
import numpy as np
import re
from collections import Counter

def normalize_text(s):
    """Normalize text for comparison (SQuAD style)"""
    s = s.lower().strip()
    # Remove punctuation
    s = re.sub(r'[^\w\s]', '', s)
    # Remove extra spaces
    s = ' '.join(s.split())
    return s

def compute_f1_text(pred_text, true_text):
    """Compute F1 score on text tokens"""
    pred_tokens = normalize_text(pred_text).split()
    true_tokens = normalize_text(true_text).split()
    
    if len(pred_tokens) == 0 or len(true_tokens) == 0:
        return 0.0
    
    common = Counter(pred_tokens) & Counter(true_tokens)
    num_same = sum(common.values())
    
    if num_same == 0:
        return 0.0
    
    precision = num_same / len(pred_tokens)
    recall = num_same / len(true_tokens)
    f1 = 2 * precision * recall / (precision + recall)
    return f1

def compute_metrics(pred):
    """
    CHARACTER-LEVEL metrics:
    1. Convert predicted token positions → character positions
    2. Extract predicted text
    3. Compare with ground truth text
    """
    start_logits, end_logits = pred.predictions
    start_preds = np.argmax(start_logits, axis=1)
    end_preds = np.argmax(end_logits, axis=1)
    
    # Get validation dataset
    val_data = pred.label_ids  # This won't work with current Trainer API
    
    # We need to pass val_dataset separately
    # For now, return token-level metrics as placeholder
    start_labels = pred.label_ids[0] if isinstance(pred.label_ids, tuple) else pred.label_ids[:, 0]
    end_labels = pred.label_ids[1] if isinstance(pred.label_ids, tuple) else pred.label_ids[:, 1]
    
    exact_match = 0
    f1_total = 0.0
    total = len(start_labels)
    
    for i in range(total):
        # Token-level comparison (will be replaced with character-level)
        if start_preds[i] == start_labels[i] and end_preds[i] == end_labels[i]:
            exact_match += 1
            f1_total += 1.0
        else:
            # Token overlap F1
            pred_tokens = set(range(start_preds[i], end_preds[i] + 1))
            true_tokens = set(range(start_labels[i], end_labels[i] + 1))
            
            overlap = pred_tokens & true_tokens
            if len(overlap) > 0:
                precision = len(overlap) / len(pred_tokens) if len(pred_tokens) > 0 else 0
                recall = len(overlap) / len(true_tokens) if len(true_tokens) > 0 else 0
                if precision + recall > 0:
                    f1 = 2 * precision * recall / (precision + recall)
                    f1_total += f1
    
    return {
        'exact_match': exact_match / total,
        'f1': f1_total / total
    }

print("✅ Metrics function defined")

✅ Metrics function defined


## 7. Model Setup

In [7]:
from transformers import AutoModelForQuestionAnswering

# MODEL_PATH defined in tokenization cell above
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_PATH)
print(f"✅ Model loaded from: {MODEL_PATH}")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
qa_outputs.weight                          | MISSING    | 
qa_outputs.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized beca

✅ Model loaded from: bert-base-uncased


## 8. Training Configuration

In [8]:
from transformers import (
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    TrainerCallback,
)
from pathlib import Path
import wandb

WANDB_PROJECT = "task3-citation-span-extraction"
ARTIFACT_NAME = "task3-bert-checkpoints"
CHECKPOINT_DIR = "/kaggle/working/checkpoints/task3_bert"


# ─── Initialize wandb ─────────────────────────
try:
    if wandb.run is None:
        wandb.init(
            project=WANDB_PROJECT,
            name="bert-base-lr3e5-batch32",
            config={
                "model": "bert-base-uncased",
                "task": "span-extraction-qa",
                "learning_rate": 3e-5,
                "batch_size": 32,
                "max_steps": 5000,
                "warmup_steps": 500,
                "weight_decay": 0.01,
            },
            resume="allow",
        )
    report_to = "wandb"
    print("✅ Wandb initialized")
except Exception as e:
    print(f"⚠️ Wandb not available: {e}")
    report_to = "none"


# ─── Auto-resume từ wandb artifact ───────────
def download_wandb_checkpoint(project, artifact_name, download_dir="/kaggle/working/resume_ckpt"):
    try:
        api = wandb.Api()
        artifact = api.artifact(f"{project}/{artifact_name}:latest", type="checkpoint")
        path = artifact.download(root=download_dir)

        checkpoints = sorted(
            Path(path).glob("checkpoint-*"),
            key=lambda x: int(x.name.split("-")[-1]),
        )

        if checkpoints:
            print(f"⬇️ Downloaded: {checkpoints[-1].name} from wandb artifact")
            return str(checkpoints[-1])

        return path

    except Exception as e:
        print(f"ℹ️ No wandb artifact to resume from: {e}")
        return None


resume_checkpoint = download_wandb_checkpoint(WANDB_PROJECT, ARTIFACT_NAME)

if not resume_checkpoint:
    print("🆕 Starting fresh training")


# ─── Callback upload checkpoint ─────────────
class WandbCheckpointCallback(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        if wandb.run is None:
            return

        ckpt_dir = Path(args.output_dir) / f"checkpoint-{state.global_step}"
        if not ckpt_dir.exists():
            return

        artifact = wandb.Artifact(
            name=ARTIFACT_NAME,
            type="checkpoint",
            metadata={
                "step": state.global_step,
                "best_metric": state.best_metric,
            },
        )

        artifact.add_dir(str(ckpt_dir))
        wandb.log_artifact(artifact)

        print(f"⬆️ checkpoint-{state.global_step} uploaded to wandb")


# ─── Training config ─────────────────────────
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    max_steps=5000,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_steps=500,
    eval_strategy="steps",
    eval_steps=500,
    logging_steps=100,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=True,
    report_to=report_to,
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=3),
        WandbCheckpointCallback(),
    ],
)

print("\n💡 Training Configuration:")
print("   Model:           bert-base-uncased")
print("   Max steps:       5,000")
print("   Effective batch: 32 (8 × 4)")
print("   Learning rate:   3e-5")
print("   Save/eval every: 500 steps → auto-upload to wandb")
print(f"   Resume from:     {resume_checkpoint or 'scratch'}")

wandb: setting up run 9pt7l848
wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260329_080244-9pt7l848
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run bert-base-lr3e5-batch32
wandb: ⭐️ View project at https://wandb.ai/tathiyennhi-hcmus/task3-citation-span-extraction
wandb: 🚀 View run at https://wandb.ai/tathiyennhi-hcmus/task3-citation-span-extraction/runs/9pt7l848


✅ Wandb initialized
ℹ️ No wandb artifact to resume from: artifact membership 'task3-bert-checkpoints:latest' not found in 'tathiyennhi-hcmus/task3-citation-span-extraction'
🆕 Starting fresh training

💡 Training Configuration:
   Model:           bert-base-uncased
   Max steps:       5,000
   Effective batch: 32 (8 × 4)
   Learning rate:   3e-5
   Save/eval every: 500 steps → auto-upload to wandb
   Resume from:     scratch


In [9]:
from transformers import (
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    TrainerCallback,
)
from pathlib import Path
from kaggle_secrets import UserSecretsClient
import wandb

WANDB_PROJECT = "task3-citation-span-extraction"
ARTIFACT_NAME = "task3-bert-checkpoints"
CHECKPOINT_DIR = "/kaggle/working/checkpoints/task3_bert"


# ─── Login wandb from Kaggle Secrets ──────────────────────────────────────────
try:
    secrets = UserSecretsClient()
    wandb_key = secrets.get_secret("WANDB_API_KEY")
    wandb.login(key=wandb_key)

    if wandb.run is None:
        wandb.init(
            project=WANDB_PROJECT,
            name="bert-base-lr3e5-batch32",
            config={
                "model": "bert-base-uncased",
                "task": "span-extraction-qa",
                "learning_rate": 3e-5,
                "batch_size": 32,
                "max_steps": 5000,
                "warmup_steps": 500,
                "weight_decay": 0.01,
            },
            resume="allow",
        )

    report_to = "wandb"
    print("✅ Wandb logged in and initialized")

except Exception as e:
    print(f"⚠️ Wandb not available: {type(e).__name__}: {e}")
    report_to = "none"


# ─── Auto-resume: download latest checkpoint from wandb artifact ─────────────
def download_wandb_checkpoint(
    project,
    artifact_name,
    download_dir="/kaggle/working/resume_ckpt",
):
    if report_to == "none":
        return None

    try:
        api = wandb.Api()
        artifact = api.artifact(
            f"{project}/{artifact_name}:latest",
            type="checkpoint",
        )
        path = artifact.download(root=download_dir)

        checkpoints = sorted(
            Path(path).glob("checkpoint-*"),
            key=lambda x: int(x.name.split("-")[-1]),
        )

        if checkpoints:
            print(f"⬇️ Downloaded: {checkpoints[-1].name} from wandb artifact")
            return str(checkpoints[-1])

        return path

    except Exception as e:
        print(f"ℹ️ No wandb artifact to resume from: {e}")
        return None


resume_checkpoint = download_wandb_checkpoint(WANDB_PROJECT, ARTIFACT_NAME)

if not resume_checkpoint:
    print("🆕 Starting fresh training")


# ─── Callback: upload each checkpoint to wandb artifact right after saving ───
class WandbCheckpointCallback(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        if report_to == "none" or wandb.run is None:
            return

        ckpt_dir = Path(args.output_dir) / f"checkpoint-{state.global_step}"
        if not ckpt_dir.exists():
            return

        artifact = wandb.Artifact(
            name=ARTIFACT_NAME,
            type="checkpoint",
            metadata={
                "step": state.global_step,
                "best_metric": state.best_metric,
            },
        )
        artifact.add_dir(str(ckpt_dir))
        wandb.log_artifact(artifact)

        print(f"⬆️ checkpoint-{state.global_step} uploaded to wandb artifact")


# ─── Training config ──────────────────────────────────────────────────────────
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    max_steps=5000,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_steps=500,
    eval_strategy="steps",
    eval_steps=500,
    logging_steps=100,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=True,
    report_to=report_to,
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=3),
        WandbCheckpointCallback(),
    ],
)

print("\n💡 Training Configuration:")
print("   Model:           BERT-base-uncased")
print("   Max steps:       5,000")
print("   Effective batch: 32 (8 × 4)")
print("   Learning rate:   3e-5")
print("   Save/eval every: 500 steps → auto-upload to wandb")
print(f"   Resume from:     {resume_checkpoint or 'scratch'}")

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


✅ Wandb logged in and initialized
ℹ️ No wandb artifact to resume from: artifact membership 'task3-bert-checkpoints:latest' not found in 'tathiyennhi-hcmus/task3-citation-span-extraction'
🆕 Starting fresh training

💡 Training Configuration:
   Model:           BERT-base-uncased
   Max steps:       5,000
   Effective batch: 32 (8 × 4)
   Learning rate:   3e-5
   Save/eval every: 500 steps → auto-upload to wandb
   Resume from:     scratch


## 9. Training Model

In [10]:
print("=" * 60)
print("🚀 TRAINING BERT FOR CITATION SPAN EXTRACTION")
print("=" * 60)

trainer.train(resume_from_checkpoint=resume_checkpoint)

print("\n✅ Training complete!")

🚀 TRAINING BERT FOR CITATION SPAN EXTRACTION


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss,Validation Loss,Exact Match,F1
500,0.361373,0.264603,0.896432,0.961836
1000,0.275821,0.205806,0.918915,0.974041
1500,0.276633,0.194213,0.921816,0.975371
2000,0.230066,0.197021,0.921671,0.978153
2500,0.201622,0.191705,0.921091,0.977372
3000,0.189983,0.191484,0.919350,0.977811
3500,0.186346,0.186375,0.924427,0.978467
4000,0.172630,0.193041,0.920366,0.977710
4500,0.158816,0.199110,0.914273,0.976324
5000,0.150826,0.195591,0.919205,0.978064


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

wandb: Adding directory to artifact (/kaggle/working/checkpoints/task3_bert/checkpoint-500)... Done. 2.8s


...

⬆️ checkpoint-4500 uploaded to wandb artifact


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

wandb: Adding directory to artifact (/kaggle/working/checkpoints/task3_bert/checkpoint-5000)... Done. 3.1s
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output

⬆️ checkpoint-5000 uploaded to wandb artifact

✅ Training complete!


## 10. Evaluate

In [11]:
print("📊 VALIDATION RESULTS")
print("=" * 60)

eval_results = trainer.evaluate()

for key, value in eval_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

print("=" * 60)
print(f"\n✅ F1 Score: {eval_results.get('eval_f1', 0):.2%}")
print(f"✅ Exact Match: {eval_results.get('eval_exact_match', 0):.2%}")

📊 VALIDATION RESULTS


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


eval_loss: 0.1864
eval_exact_match: 0.9244
eval_f1: 0.9785
eval_runtime: 121.7322
eval_samples_per_second: 56.6320
eval_steps_per_second: 1.7740
epoch: 2.5267

✅ F1 Score: 97.85%
✅ Exact Match: 92.44%


## 11. Save Model

In [12]:
final_model_path = '/kaggle/working/models/task3_bert_final'
trainer.save_model(final_model_path)
tokenizer.save_pretrained(final_model_path)

print(f"✅ Model saved to: {final_model_path}")

# Log model to wandb
try:
    wandb.save(f"{final_model_path}/*")
    print("✅ Model logged to wandb")
except:
    pass

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

wandb: WARNING Saving files without folders. If you want to preserve subdirectories pass base_path to wandb.save, i.e. wandb.save("/mnt/folder/file.h5", base_path="/mnt")
wandb: WARNING Symlinked 5 files into the W&B run directory; call wandb.save again to sync new files.


✅ Model saved to: /kaggle/working/models/task3_bert_final
✅ Model logged to wandb


## 12. Test Inference

In [13]:
import torch
from transformers import pipeline

qa_pipeline = pipeline(
    'question-answering',
    model=final_model_path,
    tokenizer=final_model_path,
    device=0 if torch.cuda.is_available() else -1
)

# Test example
test_context = "Previous studies demonstrated significant improvements in model performance. These findings support our hypothesis [CITATION_1]."
test_question = "What does citation [CITATION_1] support?"

result = qa_pipeline(
    question=test_question,
    context=test_context
)

print("\n📋 Test Inference:")
print(f"Question: {test_question}")
print(f"Context: {test_context}")
print(f"\nPredicted Answer: {result['answer']}")
print(f"Confidence: {result['score']:.4f}")
print(f"Start: {result['start']}, End: {result['end']}")

print("\n✅ BERT TRAINING COMPLETE!")

# Finish wandb run
try:
    wandb.finish()
except:
    pass

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

wandb: updating run metadata



📋 Test Inference:
Question: What does citation [CITATION_1] support?
Context: Previous studies demonstrated significant improvements in model performance. These findings support our hypothesis [CITATION_1].

Predicted Answer: These findings support our hypothesis [CITATION_1].
Confidence: 0.9913
Start: 77, End: 128

✅ BERT TRAINING COMPLETE!


wandb: uploading task3_bert_final/model.safetensors
wandb: 
wandb: Run history:
wandb:        eval/exact_match ▁▇▇▇▇▇█▇▅▇█
wandb:                 eval/f1 ▁▆▇█████▇██
wandb:               eval/loss █▃▂▂▁▁▁▂▂▂▁
wandb:            eval/runtime ▃▃▂▁▄▃▃▂▄▃█
wandb: eval/samples_per_second ▆▆▇█▅▆▆▇▅▆▁
wandb:   eval/steps_per_second ▆▆▇█▅▆▆▇▆▇▁
wandb:             train/epoch ▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
wandb:       train/global_step ▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇████
wandb:         train/grad_norm ▄█▅▃▄▃▃▄▁▁▂▁▁▂▂▂▂▂▁▁▁▃▁▁▁▂▁▃▁▁▃▁▂▂▂▂▂▂▂▁
wandb:     train/learning_rate ▂▄▅▇███▇▇▇▇▇▆▆▆▆▆▆▅▅▅▅▅▄▄▄▄▄▃▃▃▃▂▂▂▂▂▁▁▁
wandb:                      +1 ...
wandb: 
wandb: Run summary:
wandb:        eval/exact_match 0.92443
wandb:                 eval/f1 0.97847
wandb:               eval/loss 0.18639
wandb:            eval/runtime 121.7322
wandb: eval/samples_per_second 56.632
wandb:   eval/steps_per_second 1.774
wandb:              total_flos 8.359719397606195e+16
wandb:             train/e